# Canonical MathLive rebuild showcase notebook — Phase 3

This notebook is the **primary evidence surface** for this phase.

It is intentionally saved **without outputs**. Run it yourself, interact with the
widgets yourself, and decide whether the visible behavior matches the written
expectations.


## Phase goal

This phase is trying to prove one visible capability:

`IdentifierInput` in `context_or_new` now accepts canonical identifiers from
`gu_toolkit.identifiers`, including Greek names and subscripted names, while
keeping function-like names forbidden and exposing a restricted identifier-only
virtual keyboard.

This phase is **not** trying to prove a broad generic MathLive cleanup. It is
limited to the identifier field behavior described below.


## How to use this notebook

Run the cells in order.

For each demo:

- read the goal first,
- interact with the widget manually,
- open the menu manually,
- open the keyboard manually with the keyboard toggle in the field,
- rerun the nearby `...value` cell after each edit when you want to inspect the
  synchronized Python value,
- only then decide whether the checklist item should count as passing.

Do **not** treat a Python value change by itself as proof that the frontend is
correct. The visible menu, keyboard, red invalid state, and normalized display
behavior are the main evidence in this phase.


In [1]:
import import_setup
from gu_toolkit import IdentifierInput
from gu_toolkit.math_input.identifier_widget import DEFAULT_FORBIDDEN_SYMBOLS


## Context used in this phase

This phase uses one small explicit context for the main field:

- `mass`
- `theta_x`
- `alpha`

Why this matters:

- the menu should list these exact canonical names,
- selecting `theta_x` should visibly produce a subscripted identifier,
- selecting `alpha` should visibly produce a Greek letter,
- the context is small enough that you can compare the notebook text with the
  field menu and the optional context keyboard layout.


In [2]:
identifier_context = ["mass", "theta_x", "alpha"]
identifier_context


['mass', 'theta_x', 'alpha']

## Demo 1 — main `context_or_new` field

### What is this demo trying to prove?

The rebuilt `context_or_new` field should accept canonical identifiers from
`gu_toolkit.identifiers`, show them as supported MathLive display forms, and
stay visibly constrained to identifier-only actions.

### What should happen?

The field should render as a MathLive field.

When you open the field menu, you should see:

- a `Context names` submenu containing `mass`, `theta_x`, and `alpha`,
- an `Insert` submenu with a `Subscript` action,
- the same small edit-command set as before.

When you use the keyboard toggle, you should see only identifier-oriented
layouts and controls:

- context-name buttons when context names are present,
- Latin-letter buttons,
- digit buttons,
- Greek-letter buttons,
- a `sub` key for inserting a subscript placeholder,
- ordinary editing controls such as backspace, arrows, undo, redo, or hide.

You should **not** see direct function-name buttons such as `sin`, `cos`, or
`log`, and you should **not** see expression-building keys such as fractions or
roots in this identifier keyboard.

### Why should it happen?

This phase is supposed to replace the local regex-based identifier rule with the
canonical identifier layer while still keeping the visible widget surface small
and auditable.

### What would indicate the implementation is still wrong?

Failure would include any of these:

- the menu does not list the printed context names,
- the keyboard exposes broad expression-building keys,
- there is no visible way to insert a subscript placeholder,
- Greek or subscripted identifiers cannot be entered,
- typing a forbidden name such as `sin` is treated as accepted.


In [3]:
context_or_new_field = IdentifierInput(
    value="mass",
    context_names=identifier_context,
    context_policy="context_or_new",
)
context_or_new_field


## Demo 2 — manual checks for allowed identifiers and forbidden names

### What is this demo trying to prove?

Valid identifiers should be accepted, forbidden names should stay visible but
turn red, and the synchronized Python value should continue to hold the last
accepted canonical identifier.

### What should happen?

Try these by hand in the widget above, rerunning the value cell after each step
when you want to inspect the synchronized Python value:

1. Open the `Context names` submenu and select `theta_x`.
   - The field should show a visibly subscripted identifier.
   - The Python value should become `theta_x`.
2. Open the `Context names` submenu and select `alpha`.
   - The field should show a Greek identifier.
   - The Python value should become `alpha`.
3. Use the keyboard toggle, switch to a Greek layout, and insert a Greek letter.
   - The field should accept it if the resulting identifier is not forbidden.
4. Use the `sub` key from the keyboard or the `Insert -> Subscript` menu item to
   build a subscripted identifier such as `x_1`.
   - It should remain invalid while the placeholder is unfinished.
   - It should become accepted once the subscript is completed.
5. Type `sin` or `log`.
   - The field should become visibly red.
   - The Python value cell below should stay at the last accepted identifier.
6. Type `x+1`.
   - The field should become visibly red.
   - The Python value should again stay at the last accepted identifier.
7. Type plain `alpha` and then commit the edit by pressing Enter or by leaving
   the field.
   - The accepted identifier may normalize to its canonical display form after
     commit.

### Why should it happen?

This is the visible proof that the field now understands richer identifier
syntax, but still blocks forbidden function-like names.

### What would indicate the implementation is still wrong?

Failure would include any of these:

- `theta_x` cannot be entered or selected,
- Greek input is rejected,
- the subscript insertion path is missing,
- `sin` or `log` are accepted as ordinary identifiers,
- invalid drafts overwrite the last accepted Python value.


In [6]:
context_or_new_field.value


'alpha'

## Demo 3 — constructor-time forbidden-context protection

### What is this demo trying to prove?

Forbidden symbols should be rejected immediately if they are supplied through
`context_names`.

### What should happen?

Run the next cell.

It should raise a Python-side error because `sin` is forbidden in
`IdentifierInput` by default.

### Why should it happen?

The frontend red state is only for untrusted user drafts. Trusted Python-side
configuration should fail immediately instead of creating a widget that starts in
an impossible state.

### What would indicate the implementation is still wrong?

Failure would include either of these:

- the cell constructs a widget successfully even though `sin` is in the context,
- the cell fails for a reason unrelated to the forbidden context entry.


In [5]:
IdentifierInput(
    context_names=["mass", "sin"],
    context_policy="context_or_new",
)


TraitError: context_names must not contain forbidden symbols; found forbidden entry 'sin'.

## Demo 4 — custom forbidden list at construction time

### What is this demo trying to prove?

The default forbidden-symbol policy should be replaceable at construction time.

### What should happen?

The field below adds `alpha` to the default forbidden list.

Try these by hand:

- type or insert `alpha` — the field should become red,
- type or insert `theta_x` — the field should still be accepted,
- type or insert `sin` — it should still be red because it remains in the
  extended forbidden list.

### Why should it happen?

This phase needs one visible way to change the baked-in default at widget
creation without changing the general Python contract.

### What would indicate the implementation is still wrong?

Failure would include any of these:

- `alpha` is still accepted in this field,
- `theta_x` is rejected only because the forbidden list changed,
- the custom forbidden list silently replaces the field with an invalid state.


In [7]:
extended_forbidden = [*DEFAULT_FORBIDDEN_SYMBOLS, "alpha"]
custom_forbidden_field = IdentifierInput(
    value="mass",
    context_names=["mass", "theta_x"],
    context_policy="context_or_new",
    forbidden_symbols=extended_forbidden,
)
custom_forbidden_field


In [8]:
custom_forbidden_field.value


'mass'

## Manual verification checklist

Only mark an item after you have **personally observed** it in this notebook.

- [ ] The main `context_or_new` field renders.
- [ ] Its menu contains `Context names` with exactly `mass`, `theta_x`, and `alpha`.
- [ ] Its menu contains an `Insert -> Subscript` action.
- [ ] The keyboard toggle is visible for the main `context_or_new` field.
- [ ] The keyboard shows only identifier-oriented layouts and editing controls.
- [ ] The keyboard does **not** expose direct `sin`, `cos`, or `log` buttons.
- [ ] Selecting or typing `theta_x` is accepted.
- [ ] Selecting or typing `alpha` is accepted.
- [ ] A completed subscripted identifier such as `x_1` is accepted.
- [ ] An unfinished subscript placeholder stays visibly invalid.
- [ ] Typing `sin` turns the field red.
- [ ] Typing `log` turns the field red.
- [ ] Invalid drafts do **not** overwrite the last accepted Python value.
- [ ] The constructor-time forbidden-context cell raises for `sin` in `context_names`.
- [ ] The custom-forbidden field rejects `alpha` while still accepting `theta_x`.


## Short report for this phase

### Implemented

- canonical identifier validation for `IdentifierInput` via `gu_toolkit.identifiers`,
- Greek-name and subscripted canonical identifiers in `context_or_new`,
- default forbidden-symbol list for function-like names,
- constructor-time rejection of forbidden context entries,
- red invalid frontend state for forbidden or malformed drafts,
- restricted identifier-only virtual keyboard for `context_or_new`,
- subscript insertion through the keyboard and the identifier menu.

### Not implemented yet

- raw LaTeX assignment to the Python `value` trait,
- broad expression editing inside `IdentifierInput`,
- superscript support for identifier values,
- any claim that every runtime renders the keyboard identically.

### What you must verify manually

- the visible menu contents,
- the visible keyboard contents,
- the red invalid state for forbidden names,
- successful entry of Greek and subscripted identifiers,
- the last-accepted-value behavior when the visible draft is invalid.

### What remains uncertain

- exact visual details of the MathLive keyboard across notebook runtimes,
- whether every browser runtime normalizes committed accepted identifiers in the
  same way after blur or Enter.

**ready for user verification**
